# TP 7 — Modèles linéaires pour la classification

**Cours :** *Linear Models for Classification* (slides 159–189) — régression linéaire pour la classification (et son défaut de *masking*), **régression logistique** (modèle linéaire généralisé, maximum de vraisemblance par **Newton-Raphson**, cas dégénérés et régularisation, multi-classe), **SVM** (marge maximale, programme quadratique, marge souple et *hinge loss*, vecteurs de support), **classification non linéaire par noyaux**.

**À la fin de ce notebook, vous saurez :**
1. coder la régression linéaire sur indicatrices et expliquer le *masking* ;
2. coder la régression logistique binaire par **Newton-Raphson (IRLS)**, avec régularisation, puis la version **multi-classe** ;
3. résoudre le SVM par son **dual** (programme quadratique) — linéaire, à marge souple, à noyau — et par **sous-gradient** sur la *hinge loss* ;
4. comparer toutes ces méthodes sur le jeu jouet du cours (erreur de Bayes 21 %) ;
5. prédire le prochain mouvement du prix à partir du **déséquilibre du carnet d'ordres** et comprendre pourquoi, en trading, des **probabilités calibrées** (logistique) valent souvent mieux qu'une frontière (SVM).

> 📝 **Version énoncé.** Complétez les blocs `# TODO`, puis exécutez la cellule *Test* qui suit chaque exercice. Le corrigé est dans `../corriges/`.

In [ ]:
import sys, pathlib, warnings
_here = pathlib.Path.cwd()
ROOT = _here.parent if _here.name in ("exercices", "corriges") else _here
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tp_utils import *          # style graphique, check(), check_close(), timer()...
import tp_data                  # données réelles (cache / téléchargement) ou simulées

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
rng = np.random.default_rng(42)
from scipy.optimize import minimize
from scipy.special import expit, logsumexp
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import roc_auc_score, log_loss

## Partie 1 — Le cas d'école : régression linéaire pour la classification

Un classifieur est **linéaire** si ses frontières de décision sont des hyperplans. Première approche (« discriminante ») : modéliser chaque probabilité a posteriori $\delta_y(x)\approx\mathbb P(Y=y\mid X=x)$ par une fonction affine et prédire $\arg\max_y\delta_y(x)$.

Avec la matrice indicatrice $Z\in\{0,1\}^{n\times\kappa}$ ($Z_{iy} = \mathbb 1_{y_i = y}$), on fait une régression OLS **multi-sortie** : $\hat B = (X^\top X)^{-1}X^\top Z$, puis $\hat f(x) = \arg\max_y [1\ x^\top]\hat B_{\cdot y}$.

### Exercice 1.1

In [ ]:
def add_intercept(X):
    X = np.asarray(X, float).reshape(len(X), -1)
    return np.c_[np.ones(len(X)), X]


def linreg_clf_fit(X, y, n_classes):
    # TODO : Z = np.eye(n_classes)[y] ; moindres carrés de Z sur [1 X] (np.linalg.lstsq)
    raise NotImplementedError("À compléter")


def linreg_clf_scores(B, X):
    return add_intercept(X) @ B


def linreg_clf_predict(B, X):
    return linreg_clf_scores(B, X).argmax(1)

In [ ]:
# Test : sur des données linéairement séparables à 2 classes, tout va bien
X2 = np.r_[rng.normal([-2, 0], 1, (200, 2)), rng.normal([2, 0], 1, (200, 2))]; y2 = np.repeat([0, 1], 200)
B = linreg_clf_fit(X2, y2, 2)
check(np.mean(linreg_clf_predict(B, X2) == y2) > 0.95, "2 classes séparables : > 95 % de réussite")
check_close(linreg_clf_scores(B, X2).sum(1), np.ones(len(X2)), "les scores somment à 1 (mais peuvent sortir de [0, 1])")

### Expérience 1.2 — Le *masking* (slide 168)

Trois classes alignées le long de la diagonale. La classe du milieu n'est presque jamais prédite : sa fonction discriminante (affine) n'est jamais la plus grande. Le cours trouve 33 % d'erreur pour une erreur de Bayes de 2,5 %.

In [ ]:
centers = np.array([[-3, -3], [0, 0], [3, 3]])
X3 = np.vstack([rng.normal(c, 1, (100, 2)) for c in centers]); y3 = np.repeat([0, 1, 2], 100)
B3 = linreg_clf_fit(X3, y3, 3)
err3 = np.mean(linreg_clf_predict(B3, X3) != y3)
print(f"erreur de la régression linéaire : {err3:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_decision_boundary(axes[0], lambda Z: linreg_clf_predict(B3, Z), X3, y3, "Régression linéaire : la classe 1 est masquée")
t = np.linspace(-6, 6, 200); diag = np.c_[t, t]
for c in range(3):
    axes[1].plot(t, linreg_clf_scores(B3, diag)[:, c], color=PALETTE[c], label=f"δ{c}")
axes[1].set_title("Fonctions discriminantes le long de la diagonale"); axes[1].legend(); axes[1].set_xlabel("position sur la diagonale")
plt.tight_layout(); plt.show()
check(err3 > 0.2, "le masking provoque une erreur bien supérieure à l'erreur de Bayes")

## Partie 2 — Régression logistique

Modèle linéaire **généralisé** : $\mathbb P(Y=1\mid X=x) = \sigma([1\ x^\top]\beta)$, $\sigma(t) = 1/(1+e^{-t})$. La frontière $\{\sigma = 1/2\}$ est l'hyperplan $[1\ x^\top]\beta = 0$, et les probabilités restent dans $[0,1]$.

**Maximum de vraisemblance** ($z_i\in\{0,1\}$, $\tilde x_i = [1\ x_i^\top]^\top$, $p_i = \sigma(\tilde x_i^\top\beta)$) :
$$\ell(\beta) = \sum_i\big[z_i\,\tilde x_i^\top\beta - \log(1+e^{\tilde x_i^\top\beta})\big],\quad \nabla\ell = X^\top(z - p),\quad \nabla^2\ell = -X^\top WX,\ W = \mathrm{diag}(p_i(1-p_i)).$$
$\ell$ est **concave** : la méthode de **Newton-Raphson** $\beta\leftarrow\beta - (\nabla^2\ell)^{-1}\nabla\ell$ converge en quelques itérations (convergence quadratique). C'est aussi une suite de moindres carrés pondérés (*IRLS*).

**Régularisation** (cas dégénérés, classes séparables) : on maximise $\ell(\beta) - \lambda\,\beta^\top P\beta$, avec $P = \mathrm{diag}(0, 1, \dots, 1)$ (constante non pénalisée) : gradient $- 2\lambda P\beta$, Hessienne $-2\lambda P$.

### Exercice 2.1 — `logistic_newton`
Calculez $\log(1+e^t)$ avec `np.logaddexp(0, t)` (stable numériquement).

In [ ]:
def logistic_newton(X, z, lam=0.0, P=None, max_iter=50, tol=1e-10):
    """X inclut la constante. Retourne (beta, historique de la log-vraisemblance pénalisée)."""
    n, p = X.shape
    if P is None:
        P = np.eye(p); P[0, 0] = 0
    beta = np.zeros(p); hist = []
    for it in range(max_iter):
        # TODO : log-vraisemblance, gradient, Hessienne, pas de Newton
        raise NotImplementedError("À compléter")
    return beta, hist

In [ ]:
# Tests vs scikit-learn : sklearn minimise C·Σ logloss + ½||w||²  ⇔  λ = 1/(2C)
Xl = rng.normal(size=(400, 3)); zl = (rng.random(400) < expit(0.5 + Xl @ [1.5, -1, 0.3])).astype(int)
for C in [0.05, 1.0]:
    b, h = logistic_newton(add_intercept(Xl), zl, lam=1 / (2 * C))
    sk = LogisticRegression(C=C, tol=1e-12, max_iter=10000).fit(Xl, zl)
    check_close(b, np.r_[sk.intercept_, sk.coef_.ravel()], f"Newton = sklearn (C = {C})", rtol=1e-5, atol=1e-6)
check(np.all(np.diff(h) >= -1e-9) and len(h) < 12, f"vraisemblance croissante, convergence en {len(h)} itérations")

### Expérience 2.2 — Classes séparables : la vraisemblance n'a pas de maximum

Si les classes sont linéairement séparables, on peut toujours augmenter $\ell$ en multipliant $\beta$ par un scalaire > 1 : $\|\hat\beta\|\to\infty$ et la Hessienne devient singulière. La régularisation rend le problème bien posé.

In [ ]:
Xs = np.r_[rng.normal([-2, 0], 0.7, (50, 2)), rng.normal([2, 0], 0.7, (50, 2))]; zs = np.repeat([0, 1], 50)
norms = {}
for lam in [0.0, 0.1]:
    nb = []
    for k in range(1, 30):
        try:
            b, _ = logistic_newton(add_intercept(Xs), zs, lam=lam, max_iter=k, tol=0)
            nb.append(np.linalg.norm(b))
        except np.linalg.LinAlgError:
            break
    norms[lam] = nb
fig, ax = plt.subplots(figsize=(7, 3.8))
for i, (lam, nb) in enumerate(norms.items()):
    ax.plot(range(1, len(nb) + 1), nb, marker="o", ms=3, color=PALETTE[i], label=f"λ = {lam}")
ax.set_xlabel("itérations de Newton"); ax.set_ylabel("‖β‖"); ax.legend(); ax.set_yscale("log")
ax.set_title("Classes séparables : ‖β‖ diverge sans régularisation")
plt.show()

### Exercice 2.3 — Régression logistique multi-classe

Avec une **classe de référence** $\kappa$ : $\log\frac{\mathbb P(Y=y\mid x)}{\mathbb P(Y=\kappa\mid x)} = \tilde x^\top\beta_y$ pour $y<\kappa$, d'où
$$\delta_y(x) = \frac{e^{\tilde x^\top\beta_y}}{1+\sum_{z<\kappa}e^{\tilde x^\top\beta_z}},\qquad \delta_\kappa(x) = \frac{1}{1+\sum_{z<\kappa}e^{\tilde x^\top\beta_z}}.$$
C'est un *softmax* des scores $(\tilde x^\top\beta_1,\dots,\tilde x^\top\beta_{\kappa-1}, 0)$. Paramètres : $B\in\mathbb R^{(d+1)\times(\kappa-1)}$. On écrit la **log-vraisemblance négative** et son **gradient** $X^\top(\Pi - Z)$ restreint aux $\kappa-1$ premières colonnes ($\Pi$ = probabilités, $Z$ = indicatrices) ; l'optimisation est confiée à L-BFGS.

In [ ]:
def multinomial_nll_grad(b_flat, X, y, K):
    """X inclut la constante ; y dans {0..K-1}, classe de référence = K-1."""
    p = X.shape[1]
    B = b_flat.reshape(p, K - 1)
    # TODO : scores S = [X B, 0] ; log-softmax via logsumexp ; gradient = Xᵀ(Π − Z) restreint aux K − 1 premières colonnes
    raise NotImplementedError("À compléter")


def multinomial_fit(X, y, K):
    Xi = add_intercept(X)
    res = minimize(multinomial_nll_grad, np.zeros(Xi.shape[1] * (K - 1)), args=(Xi, y, K),
                   jac=True, method="L-BFGS-B", options={"maxiter": 5000, "gtol": 1e-10})
    return res.x.reshape(Xi.shape[1], K - 1)


def multinomial_proba(B, X):
    S = np.c_[add_intercept(X) @ B, np.zeros(len(X))]
    return np.exp(S - logsumexp(S, axis=1)[:, None])

In [ ]:
# Tests : gradient par différences finies, puis comparaison des probabilités avec sklearn (sans pénalité)
Xm = np.vstack([rng.normal(c, 1.5, (150, 2)) for c in centers]); ym = np.repeat([0, 1, 2], 150)
b0 = rng.normal(size=3 * 2) * 0.1
f0, g0 = multinomial_nll_grad(b0, add_intercept(Xm), ym, 3)
num = np.array([(multinomial_nll_grad(b0 + 1e-6 * e, add_intercept(Xm), ym, 3)[0] - f0) / 1e-6 for e in np.eye(6)])
check_close(g0, num, "gradient = différences finies", rtol=1e-4, atol=1e-3)
Bm = multinomial_fit(Xm, ym, 3)
skm = LogisticRegression(C=1e8, tol=1e-12, max_iter=100000).fit(Xm, ym)
check_close(multinomial_proba(Bm, Xm), skm.predict_proba(Xm), "probabilités = sklearn multinomial", atol=1e-4)
check(np.mean(multinomial_proba(Bm, X3).argmax(1) == y3) > 0.9, "pas de masking avec la logistique multi-classe")

## Partie 3 — Support Vector Machines

**Principe :** construire explicitement l'hyperplan séparateur de **marge maximale**. Avec $y_i\in\{-1, 1\}$ et la convention du cours $f(x) = x^\top\beta - \beta_0$ :
- **marge dure** : $\min\|\beta\|^2$ s.c. $y_i(x_i^\top\beta - \beta_0)\ge1$ (la marge vaut $1/\|\beta\|$) ;
- **marge souple** : $\min\|\beta\|^2 + C\sum_i\xi_i$ s.c. $y_i(x_i^\top\beta - \beta_0)\ge1-\xi_i$, $\xi_i\ge0$ — équivalent à minimiser $\|\beta\|^2 + C\sum_i\max\{0, 1 - y_if(x_i)\}$ (*hinge loss*). Le cours note ce paramètre λ ; grand $C$ = on privilégie les contraintes (petite marge).

**Dual** (programme quadratique) : $\max_\alpha \sum_i\alpha_i - \frac12\sum_{i,j}\alpha_i\alpha_jy_iy_j\,x_i^\top x_j$ s.c. $0\le\alpha_i\le C$, $\sum_i\alpha_iy_i = 0$. Alors $\beta = \sum_i\alpha_iy_ix_i$ ; seuls les **vecteurs de support** ($\alpha_i>0$) comptent. Les produits scalaires $x_i^\top x_j$ peuvent être remplacés par un **noyau** $k(x_i, x_j)$.

### Exercice 3.1 — SVM par le dual
On confie le QP au solveur générique SLSQP de SciPy (fourni : à vous d'écrire l'objectif et son gradient). Puis :
- vecteurs de support : $\alpha_i > 10^{-6}C$ ; vecteurs **sur la marge** : $10^{-6}C<\alpha_i<(1-10^{-6})C$ ;
- $\beta_0$ : pour un vecteur sur la marge, $y_if(x_i) = 1$, d'où $\beta_0 = \sum_j\alpha_jy_jk(x_j, x_i) - y_i$ (on moyenne sur ces vecteurs) ;
- $f(x) = \sum_j\alpha_jy_jk(x_j, x) - \beta_0$.

In [ ]:
def linear_kernel(A, B):
    return np.asarray(A, float) @ np.asarray(B, float).T


def rbf_kernel(A, B, sigma=1.0):
    A = np.asarray(A, float); B = np.asarray(B, float)
    D2 = (A ** 2).sum(1)[:, None] - 2 * A @ B.T + (B ** 2).sum(1)[None, :]
    return np.exp(-np.maximum(D2, 0) / (2 * sigma ** 2))


def poly_kernel(A, B, degree=4, c=1.0):
    return (np.asarray(A, float) @ np.asarray(B, float).T + c) ** degree


def svm_dual_fit(X, y, C=1.0, kernel=linear_kernel):
    """y dans {-1, 1}. Retourne un dict (alpha, beta0, X, y, kernel)."""
    y = np.asarray(y, float); n = len(y)
    K = kernel(X, X)
    Q = (y[:, None] * y[None, :]) * K
    # TODO : objectif dual (à minimiser : ½ αᵀQα − Σα) et son gradient
    raise NotImplementedError("À compléter")
    cons = {"type": "eq", "fun": lambda a: a @ y, "jac": lambda a: y}
    res = minimize(fun, np.zeros(n), jac=jac, bounds=[(0, C)] * n, constraints=[cons],
                   method="SLSQP", options={"maxiter": 2000, "ftol": 1e-12})
    a = res.x
    # TODO : β0 par moyenne sur les vecteurs de support sur la marge
    raise NotImplementedError("À compléter")
    return {"alpha": a, "beta0": beta0, "X": np.asarray(X, float), "y": y, "kernel": kernel}


def svm_decision(model, Xq):
    sv = model["alpha"] > 1e-8
    return model["kernel"](Xq, model["X"][sv]) @ (model["alpha"][sv] * model["y"][sv]) - model["beta0"]

In [ ]:
# Tests vs scikit-learn (sklearn : f(x) = Σ α y k + b, donc b = −β0)
mix = tp_data.ESLMixture()
Xe, ye = mix.sample(100, seed=1); ye_pm = 2 * ye - 1
Xq = rng.uniform(-2, 3, (30, 2))
for name, ker, skker in [("linéaire", linear_kernel, dict(kernel="linear")),
                         ("gaussien σ=1", lambda A, B: rbf_kernel(A, B, 1.0), dict(kernel="rbf", gamma=0.5))]:
    mdl = svm_dual_fit(Xe, ye_pm, C=1.0, kernel=ker)
    sks = SVC(C=1.0, tol=1e-8, **skker).fit(Xe, ye_pm)
    check_close(svm_decision(mdl, Xq), sks.decision_function(Xq), f"SVM {name} : f(x) = sklearn", rtol=1e-3, atol=1e-3)
    check(set(np.where(mdl["alpha"] > 1e-5)[0]) == set(sks.support_), f"SVM {name} : mêmes vecteurs de support")

### Expérience 3.2 — Marge dure, marge souple, vecteurs de support

In [ ]:
Xh = np.r_[rng.normal([-1.5, -1], 0.6, (40, 2)), rng.normal([1.5, 1], 0.6, (40, 2))]; yh = np.repeat([-1, 1], 40)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, C in zip(axes, [1e4, 1.0, 0.02]):
    mdl = svm_dual_fit(Xh, yh, C=C)
    xx, yy = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-3.5, 3.5, 200))
    F = svm_decision(mdl, np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contour(xx, yy, F, levels=[-1, 0, 1], colors=[INK2, INK, INK2], linestyles=["--", "-", "--"], linewidths=1)
    for i, c in enumerate([-1, 1]):
        ax.scatter(*Xh[yh == c].T, s=18, color=PALETTE[i], label=f"y = {c}")
    sv = mdl["alpha"] > 1e-6 * C
    ax.scatter(*Xh[sv].T, s=80, facecolors="none", edgecolors=INK, linewidths=1, label="vecteurs de support")
    beta = (mdl["alpha"] * yh) @ Xh
    ax.set_title(f"C = {C:g} : marge 1/‖β‖ = {1 / np.linalg.norm(beta):.2f}, {sv.sum()} VS")
axes[0].legend(loc="lower right"); plt.tight_layout(); plt.show()

### Exercice 3.3 — Le SVM linéaire par sous-gradient (vue « perte hinge »)

Le dual coûte au moins $O(n^2)$ en mémoire. Pour un SVM **linéaire** sur beaucoup de données, on minimise directement
$J(\beta,\beta_0) = \frac\lambda2\|\beta\|^2 + \frac1n\sum_i\max\{0, 1 - y_i(x_i^\top\beta - \beta_0)\}$ par **descente de sous-gradient** : sur les points actifs $\mathcal A = \{i : y_if(x_i) < 1\}$,
$\partial_\beta J = \lambda\beta - \frac1n\sum_{i\in\mathcal A}y_ix_i$, $\ \partial_{\beta_0}J = \frac1n\sum_{i\in\mathcal A}y_i$, avec un pas $\eta_t = 1/(\lambda t)$ et la **moyenne** des itérés de la seconde moitié (algorithme de type *Pegasos*). C'est la même perte que la marge souple, avec $\lambda = 2/(nC)$.

In [ ]:
def linear_svm_subgradient(X, y, lam=0.01, n_iter=3000):
    X = np.asarray(X, float); y = np.asarray(y, float); n, d = X.shape
    beta, b0 = np.zeros(d), 0.0
    avg_b, avg_b0, cnt = np.zeros(d), 0.0, 0
    for t in range(1, n_iter + 1):
        # TODO : points actifs : y(Xβ − β0) < 1 ; sous-gradients de l'énoncé ; pas 1/(λt) ; moyenne des itérés de la 2e moitié
        raise NotImplementedError("À compléter")
    return avg_b / cnt, avg_b0 / cnt

In [ ]:
# Test : même performance que LinearSVC (même perte)
Xbig = np.r_[rng.normal([-1, -0.5], 1.2, (3000, 2)), rng.normal([1, 0.5], 1.2, (3000, 2))]; ybig = np.repeat([-1, 1], 3000)
bb, bb0 = linear_svm_subgradient(Xbig, ybig, lam=0.01)
acc_ours = np.mean(np.sign(Xbig @ bb - bb0) == ybig)
acc_sk = LinearSVC(C=2 / (len(ybig) * 0.01), loss="hinge", max_iter=100000).fit(Xbig, ybig).score(Xbig, ybig)
check(abs(acc_ours - acc_sk) < 0.01, f"sous-gradient ({acc_ours:.3f}) ≈ LinearSVC ({acc_sk:.3f})")

## Partie 4 — Classification non linéaire par noyaux, et grand comparatif

Remplacer $x_i^\top x_j$ par $k(x_i, x_j)$ revient à chercher un hyperplan dans un espace de caractéristiques (éventuellement de dimension infinie) : la frontière dans l'espace d'origine devient non linéaire. Pour la **régression logistique à noyau**, le théorème du représentant donne $f(x) = \sum_j\alpha_jk(x_j, x)$ et la pénalité $\|f\|_{\mathcal H}^2 = \alpha^\top K\alpha$ : c'est exactement `logistic_newton` avec le design $[1\ K]$ et la matrice de pénalité $P = \mathrm{diag}(0, K)$.

### Exercice 4.1 — Régression logistique à noyau gaussien

In [ ]:
def kernel_logistic_fit(X, z, sigma=1.0, lam=0.01):
    # TODO : design [1, K] et pénalité P = diag(0, K + 1e-8 I), puis logistic_newton ; renvoyer un dict (coef, X, sigma)
    raise NotImplementedError("À compléter")


def kernel_logistic_proba(model, Xq):
    K = rbf_kernel(Xq, model["X"], model["sigma"])
    return expit(model["coef"][0] + K @ model["coef"][1:])

In [ ]:
# Le comparatif du cours : mélange de gaussiennes, n = 100 + 100, erreur de Bayes ≈ 21 %
Xt, yt = mix.sample(5000, seed=2)
models = {}
B = linreg_clf_fit(Xe, ye, 2);                              models["régression linéaire"] = lambda Z: linreg_clf_predict(B, Z)
bl, _ = logistic_newton(add_intercept(Xe), ye, lam=1e-3);   models["logistique"] = lambda Z: (add_intercept(Z) @ bl > 0).astype(int)
s_lin = svm_dual_fit(Xe, ye_pm, C=1.0);                     models["SVM linéaire (C = 1)"] = lambda Z: (svm_decision(s_lin, Z) > 0).astype(int)
s_pol = svm_dual_fit(Xe, ye_pm, C=1.0, kernel=lambda A, B_: poly_kernel(A, B_, 4))
models["SVM noyau polynomial (deg 4)"] = lambda Z: (svm_decision(s_pol, Z) > 0).astype(int)
s_rbf = svm_dual_fit(Xe, ye_pm, C=1.0, kernel=lambda A, B_: rbf_kernel(A, B_, 0.7))
models["SVM noyau gaussien (σ = 0,7)"] = lambda Z: (svm_decision(s_rbf, Z) > 0).astype(int)
klr = kernel_logistic_fit(Xe, ye, sigma=0.7, lam=0.01)
models["logistique noyau gaussien"] = lambda Z: (kernel_logistic_proba(klr, Z) > 0.5).astype(int)
models["Bayes"] = mix.bayes_predict

table = pd.Series({k: np.mean(f(Xt) != yt) for k, f in models.items()}, name="erreur de test").to_frame()
check(table.loc["SVM noyau gaussien (σ = 0,7)", "erreur de test"] < table.loc["SVM linéaire (C = 1)", "erreur de test"],
      "le noyau gaussien bat le SVM linéaire sur ce problème non linéaire")
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, k in zip(axes.ravel(), ["logistique", "SVM linéaire (C = 1)", "SVM noyau polynomial (deg 4)",
                                "SVM noyau gaussien (σ = 0,7)", "logistique noyau gaussien", "Bayes"]):
    plot_decision_boundary(ax, models[k], Xe, ye, f"{k} — {table.loc[k, 'erreur de test']:.1%}")
plt.tight_layout(); plt.show()
(100 * table).round(1).rename(columns={"erreur de test": "erreur de test (%)"})

## Partie 5 — Application : déséquilibre du carnet d'ordres et prochain mouvement de prix

En trading haute fréquence, une variable très étudiée est le **déséquilibre du carnet** au meilleur prix : $I = \frac{Q_{\text{bid}} - Q_{\text{ask}}}{Q_{\text{bid}} + Q_{\text{ask}}}\in[-1, 1]$. Si beaucoup plus de volume attend à l'achat qu'à la vente, le prochain mouvement du prix médian a plus de chances d'être à la hausse.

**Simulateur** (inspiré des modèles empiriques de la littérature de microstructure) : variables $I$, flux d'ordres récent signé normalisé `flow`, fourchette `spread` (1, 2 ou 3 *ticks*). Vrai modèle : $\mathbb P(\text{hausse}) = \sigma\big((2.5\,I + 1.5\,I^3 + 0.5\,\text{flow})/\text{spread}\big)$ — saturation en $I$ et signal dilué quand la fourchette est large.

In [ ]:
def simulate_lob(n, seed):
    r_ = np.random.default_rng(seed)
    I = 2 * r_.beta(2, 2, n) - 1
    flow = 0.6 * I + 0.8 * r_.standard_normal(n)
    spread = r_.choice([1, 2, 3], n, p=[0.6, 0.3, 0.1])
    logit = (2.5 * I + 1.5 * I ** 3 + 0.5 * flow) / spread
    up = (r_.random(n) < expit(logit)).astype(int)
    return pd.DataFrame({"I": I, "flow": flow, "spread": spread, "up": up}), expit(logit)

lob_tr, _ = simulate_lob(4000, seed=10)
lob_te, p_true_te = simulate_lob(50000, seed=11)
lob_tr.head()

### Exercice 5.1 — Trois modèles
1. **logistique brute** sur $(I, \text{flow}, \text{spread})$ ;
2. **logistique avec variables construites** : $(I/\text{spread},\ I^3/\text{spread},\ \text{flow}/\text{spread})$ — *feature engineering* guidé par la connaissance du marché ;
3. **SVM à noyau gaussien** sur les variables brutes standardisées (pour $n = 4000$, on utilise `sklearn.svm.SVC`, qui résout exactement le même dual que notre `svm_dual_fit` avec un solveur SMO plus rapide).

Comparez l'*accuracy*, l'AUC, et pour les modèles probabilistes la **log-loss** (vraisemblance hors échantillon).

In [ ]:
def raw(df):
    return df[["I", "flow", "spread"]].to_numpy(float)

def engineered(df):
    s = df["spread"].to_numpy(float)
    return np.c_[df["I"] / s, df["I"] ** 3 / s, df["flow"] / s]

ztr, zte = lob_tr["up"].to_numpy(), lob_te["up"].to_numpy()
# TODO : logistic_newton sur [1, raw] puis [1, engineered] (λ = 1e-3) ; SVC(C=1, rbf, gamma=0.5) sur les variables brutes standardisées → p_raw, p_eng, f_svm
raise NotImplementedError("À compléter")
res = pd.DataFrame({
    "accuracy": [np.mean((p_raw > .5) == zte), np.mean((p_eng > .5) == zte), np.mean((f_svm > 0) == zte), np.mean((p_true_te > .5) == zte)],
    "AUC": [roc_auc_score(zte, p_raw), roc_auc_score(zte, p_eng), roc_auc_score(zte, f_svm), roc_auc_score(zte, p_true_te)],
    "log-loss": [log_loss(zte, p_raw), log_loss(zte, p_eng), np.nan, log_loss(zte, p_true_te)]},
    index=["logistique brute", "logistique + variables construites", "SVM gaussien", "vraie probabilité (oracle)"])
check(res.loc["logistique + variables construites", "log-loss"] < res.loc["logistique brute", "log-loss"],
      "les variables construites améliorent la vraisemblance hors échantillon")
res

### Expérience 5.2 — Calibration : les probabilités annoncées sont-elles justes ?

Un modèle est **calibré** si, parmi les situations où il annonce 70 % de hausse, la hausse se produit bien 70 % du temps. On regroupe les prédictions par tranches (quantiles) et on compare fréquence prédite et fréquence observée — **séparément selon la fourchette**, car un modèle peut être calibré en moyenne mais faux dans chaque situation. Le SVM, lui, ne produit pas de probabilité : son score $f(x)$ n'a pas d'échelle naturelle (il faudrait un recalibrage *a posteriori*, type Platt).

In [ ]:
def reliability(p, z, n_bins=8):
    edges = np.quantile(p, np.linspace(0, 1, n_bins + 1))
    idx = np.clip(np.searchsorted(edges, p, side="right") - 1, 0, n_bins - 1)
    return np.array([p[idx == b].mean() for b in range(n_bins)]), np.array([z[idx == b].mean() for b in range(n_bins)])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, sp in zip(axes[:2], [1, 3]):
    m = lob_te["spread"].to_numpy() == sp
    for i, (name, p) in enumerate([("logistique brute", p_raw), ("logistique + variables construites", p_eng)]):
        pm, om = reliability(p[m], zte[m])
        ax.plot(pm, om, marker="o", color=PALETTE[i], label=name)
    ax.plot([0, 1], [0, 1], color=GRAY, ls="--", lw=1, label="calibration parfaite")
    ax.set_xlabel("probabilité prédite"); ax.set_ylabel("fréquence observée")
    ax.set_title(f"Fiabilité quand la fourchette = {sp} tick{'s' if sp > 1 else ''}")
axes[0].legend(loc="upper left")
axes[2].scatter(f_svm[:3000], p_true_te[:3000], s=4, color=AQUA, alpha=0.5)
axes[2].set_xlabel("score SVM f(x)"); axes[2].set_ylabel("vraie probabilité de hausse")
axes[2].set_title("SVM : un score monotone, pas une probabilité")
plt.tight_layout(); plt.show()

**Questions.**
1. Le SVM gaussien a une *accuracy* et une AUC proches de la logistique avec variables construites. Pourquoi un desk de trading préférera-t-il quand même la logistique ?
2. Que gagne-t-on avec les variables construites par rapport à la logistique brute ?

> ✍️ *Votre réponse :* …

## Partie 6 — Synthèse : quel classifieur linéaire, quand ?

| Méthode | Ce qu'elle modélise | Ajustement | Forces | Faiblesses | Quand l'utiliser |
|---|---|---|---|---|---|
| Régression linéaire sur indicatrices | $\mathbb P(Y=y\mid x)$ affine | OLS (forme close) | trivial, rapide | scores hors $[0,1]$, **masking** dès 3 classes | jamais en pratique (référence pédagogique) |
| **Logistique** (binaire / multi-classe) | log-odds affines | Newton / IRLS (concave), L-BFGS | **probabilités calibrées**, interprétable, inférence, rapide | frontière linéaire ; diverge si séparable (→ régulariser) | scoring, probabilités pour décider/dimensionner (trading, crédit), *baseline* |
| **SVM** linéaire (marge souple) | hyperplan de marge maximale | QP dual, ou sous-gradient (hinge) | robuste (seuls les VS comptent), bonne généralisation, grande dimension | pas de probabilité, C à régler | classification pure, texte/haute dimension |
| **SVM à noyau** | frontière non linéaire | QP dual ($O(n^2)$ mémoire) | très flexible, excellent sur petits/moyens jeux | coût en $n$, choix du noyau/σ/C, pas de probabilité | $n$ ≲ 10⁴–10⁵, frontière non linéaire |
| Logistique à noyau | log-odds non linéaires | Newton sur $n$ paramètres | flexible **et** probabiliste | coût $O(n^3)$, solution non parcimonieuse | petits jeux, besoin de probabilités non linéaires |

Les pertes associées (0-1, quadratique, hinge, logistique, exponentielle) sont comparées au début du TP 5.